Pôvodná verzia algoritmu, ktorá sa nachádza v notebooku **1_Algorithmic_mask_creation**, bola navrhnutá pre spektrogramy s menej výraznými farbami a stabilným rozložením frekvenčných línií. Po zmene dát, ktoré začal poskytovať doménový expert, sa však ukázalo, že pôvodný prístup už nie je dostatočný.

## Import knižníc
V tejto bunke importujeme všetky potrebné knižnice.

In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

## Vymazanie starých masiek z výstupného priečinka

Tento blok zabezpečí, že priečinok s maskami bude pred spracovaním prázdny.
Predchádza to miešaniu starých a nových výsledkov.

In [3]:
folder = 'data/Algorithmic_mask_creation/creat_mask_zasielka6'

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)

## Príprava priečinkov

In [4]:
input_folder = 'data/CropPictures/orezane_zasielka6_new'
output_folder = 'data/Algorithmic_mask_creation/creat_mask_zasielka6'
os.makedirs(output_folder, exist_ok=True)
filenames = sorted([f for f in os.listdir(input_folder) if f.endswith('.jpg')])


### Funkcia `connect_line_segment()`

Táto funkcia slúži na **spojenie dvoch bodov priamkou**, ale iba v rámci definovaného vertikálneho segmentu (`seg_top` až `seg_bot`).  
Používa sa pri opravovaní horizontálnych medzier v maskách, kde je potrebné doplniť chýbajúci úsek línie, ale zároveň nechceme zasahovať mimo príslušného frekvenčného pásma.

**Čo robí:**
- vytvorí dočasnú prázdnu masku,
- nakreslí priamku medzi bodmi `p1` a `p2`,
- oreže ju tak, aby existovala iba v danom vertikálnom rozsahu,
- spojí ju s pôvodnou maskou pomocou bitwise OR.


In [5]:
def connect_line_segment(full_mask, p1, p2, seg_top, seg_bot, thickness=5):
    temp = np.zeros_like(full_mask)
    cv2.line(temp, p1, p2, 255, thickness)
    temp[:seg_top, :] = 0
    temp[seg_bot + 1:, :] = 0
    return cv2.bitwise_or(full_mask, temp)

### Funkcia `connect_z_shape_dynamic()`

Táto funkcia vytvára **Z‑tvarové premostenie** medzi dvoma bodmi.  
Používa sa pri **veľkých horizontálnych medzerách**, kde jednoduchá priamka nestačí.

**Ako funguje:**
- vypočíta horizontálny offset podľa vzdialenosti bodov,
- určí výšku „Z‑tvaru“ podľa veľkosti medzery,
- vytvorí štyri kontrolné body (p1 → horný ohyb → dolný ohyb → p2),
- nakreslí tri úsečky, ktoré vytvoria tvar písmena Z.

Používa sa len pri medzerách nad definovaným prahom (`z_threshold`).


In [6]:
def connect_z_shape_dynamic(mask, p1, p2, offset_ratio=0.05, max_z_height=180, min_z_height=30, gap_length=None):
    x1, y1 = p1
    x2, y2 = p2
    dx = x2 - x1
    offset = int(abs(dx) * offset_ratio)
    height = int(gap_length / 2) if gap_length else int(abs(dx) / 2)
    height = min(max(height, min_z_height), max_z_height)
    mid1_x = x1 + offset if dx >= 0 else x1 - offset
    mid2_x = x2 - offset if dx >= 0 else x2 + offset
    mid_y = int((y1 + y2) / 2)
    z_points = [(x1, y1), (mid1_x, mid_y - height), (mid2_x, mid_y + height), (x2, y2)]
    for i in range(len(z_points) - 1):
        cv2.line(mask, z_points[i], z_points[i + 1], 255, thickness=2)
    return mask

Funkcia `gaps_overlap`:
- Kontroluje, či sa dva horizontálne gapy prekrývajú.

### Funkcia `gaps_overlap()`

Táto funkcia zisťuje, či sa **dve horizontálne medzery** (gapy) nachádzajú približne na rovnakej pozícii v osi X.

**Ako funguje:**
- vypočíta prekryv dvoch intervalov,
- normalizuje ho podľa dĺžky oboch gapov,
- rozhodne, či prekryv presahuje minimálny pomer (`min_overlap_ratio`).

Používa sa pri zoskupovaní medzier pred opravou.


In [7]:
def gaps_overlap(g1, g2, min_overlap_ratio=0.2):
    start1, end1 = g1['start'], g1['end']
    start2, end2 = g2['start'], g2['end']
    inter_start = max(start1, start2)
    inter_end = min(end1, end2)
    overlap = max(0, inter_end - inter_start)
    len1 = end1 - start1
    len2 = end2 - start2
    r1 = overlap / len1 if len1 > 0 else 0
    r2 = overlap / len2 if len2 > 0 else 0
    return r1 >= min_overlap_ratio and r2 >= min_overlap_ratio

### Funkcia `extend_line_ends_near_edges()`

Táto funkcia slúži na **automatické predlžovanie segmentovaných línií smerom k ľavému alebo pravému okraju obrázka**, ak sa končia príliš blízko okraja.  

**Hlavné kroky:**
- Pre každý detegovaný vertikálny pás (row range) sa extrahujú komponenty.
- Identifikujú sa:
   - najľavejšia významná komponenta,
   - najpravšia významná komponenta.
- Ak je komponenta bližšie k okraju než `edge_max_gap`, nakreslí sa krátka horizontálna čiara:
   - od okraja → k najbližšiemu bodu komponenty.
- Predĺženie sa vykonáva len pre komponenty s dostatočnou plochou (`min_area`), aby sa predišlo predlžovaniu šumu.

Používa sa pred opravou veľkých medzier.


In [8]:
def extend_line_ends_near_edges(mask, row_ranges=None, edge_max_gap=20, endpoint_band=5, thickness=2, min_area=30):
    out = mask.copy()
    H, W = mask.shape
    if row_ranges is None:
        row_ranges = find_row_ranges(mask, max_lines=3)

    for (ymin, ymax) in row_ranges:
        row = mask[ymin:ymax+1, :]
        m = (row > 0).astype(np.uint8)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
        if num_labels <= 1:
            continue

        left_idx, right_idx = None, None
        left_x_min, right_x_max = W + 1, -1

        for lbl in range(1, num_labels):
            x, y, bw, bh, area = stats[lbl]
            if area < min_area:
                continue
            x_min = x
            x_max = x + bw - 1
            if x_min < left_x_min:
                left_x_min, left_idx = x_min, lbl
            if x_max > right_x_max:
                right_x_max, right_idx = x_max, lbl

        if left_idx is not None and 0 < left_x_min <= edge_max_gap:
            ys, xs = np.where(labels == left_idx)
            band = xs <= (left_x_min + endpoint_band)
            yL = int(np.median(ys[band])) if np.any(band) else int(np.median(ys))
            cv2.line(out, (0, yL + ymin), (left_x_min, yL + ymin), 255, thickness)

        if right_idx is not None and (W - 1 - right_x_max) <= edge_max_gap and right_x_max < W-1:
            ys, xs = np.where(labels == right_idx)
            band = xs >= (right_x_max - endpoint_band)
            yR = int(np.median(ys[band])) if np.any(band) else int(np.median(ys))
            cv2.line(out, (right_x_max, yR + ymin), (W - 1, yR + ymin), 255, thickness)

    return out

### Funkcia `remove_linear_outliers()`

Táto funkcia odstraňuje **malé, nesúvislé alebo náhodné pixely**, ktoré sa síce nachádzajú v horizontálnom smere, ale nepredstavujú skutočné Spread‑F línie.

Používa kombináciu:
- **hustotného filtra** (kontroluje, či je v okolí dostatok pixlov),
- **dilatácie** (rozšírenie jadra línie),
- **distance transform** (meria vzdialenosť od „hlavnej línie“),
- **connected components** (vyhodnocuje každý objekt samostatne).

**Logika:**
- ak je komponenta príliš ďaleko od hustého jadra línie → odstráni sa,
- ak je blízko → ponechá sa.

**Výsledok:**
- odstránenie šumu,
- zachovanie len tých komponentov, ktoré majú charakter horizontálnej línie,
- stabilnejší vstup pre ďalšie kroky pipeline.


In [9]:
def remove_linear_outliers(binary_mask, line_len=99, density_thr=0.32, v_radius=6, h_spread=151, v_thick=1):
    m = (binary_mask > 0).astype(np.uint8)
    k = np.ones((1, int(line_len)), np.float32)
    dens = cv2.filter2D(m.astype(np.float32), -1, k, borderType=cv2.BORDER_REPLICATE) / float(line_len)
    core = (dens >= float(density_thr)).astype(np.uint8) * 255
    if v_thick > 0:
        core = cv2.dilate(core, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 2*v_thick+1)), 1)
    if h_spread > 0:
        core = cv2.dilate(core, cv2.getStructuringElement(cv2.MORPH_RECT, (int(h_spread), 1)), 1)
    dist = cv2.distanceTransform(255 - core, cv2.DIST_L2, 3)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    out = np.zeros_like(binary_mask)
    for lbl in range(1, num):
        comp = (labels == lbl)
        if comp.any() and dist[comp].min() <= float(v_radius):
            out[comp] = 255
    return out

### Funkcia `filter_components_by_shape()`

Táto funkcia filtruje komponenty podľa ich **tvaru, veľkosti a orientácie**.  
Je to dôležité, pretože Spread‑F línie sú:
- pretiahnuté,
- prevažne horizontálne,
- s malým uhlovým sklonom.

**Funkcia vyraďuje:**
- malé bodové objekty (šum),
- vertikálne alebo diagonálne štruktúry,
- objekty s nízkou elongáciou (teda nie sú „línie“).

**Používané metriky:**
- plocha komponenty,
- kovariančná matica → vlastné čísla → elongácia,
- uhol hlavnej osi komponenty.

**Výsledok:**
- maska obsahuje len objekty, ktoré sa tvarovo podobajú Spread‑F líniám.


In [10]:
def filter_components_by_shape(mask, min_area=12, large_keep=600, min_elong=3.0, max_angle_deg=20.0):
    m = (mask > 0).astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    out = np.zeros_like(mask)
    for lbl in range(1, num):
        area = int(stats[lbl, cv2.CC_STAT_AREA])
        if area >= large_keep:
            out[labels == lbl] = 255
            continue
        if area < min_area:
            continue
        ys, xs = np.where(labels == lbl)
        if xs.size < 5:
            continue
        xs = xs.astype(np.float32); ys = ys.astype(np.float32)
        C = np.cov(np.vstack([xs, ys]))
        vals, vecs = np.linalg.eigh(C)
        lam_min, lam_max = float(vals[0]), float(vals[1])
        elong = (np.inf if lam_min <= 1e-6 else lam_max / lam_min)
        vx, vy = vecs[0,1], vecs[1,1]
        ang = abs(np.degrees(np.arctan2(vy, vx)))
        if ang > 90.0:
            ang = 180.0 - ang
        if (elong >= min_elong) and (ang <= max_angle_deg):
            out[labels == lbl] = 255
    return out

### Funkcia `keep_long_horizontal_runs()`

Táto funkcia zachováva len **dlhé horizontálne úseky**, ktoré sú typické pre Spread‑F štruktúry.  
Krátke alebo prerušované úseky sa odstránia.

**Ako to funguje:**
- najprv sa malé medzery horizontálne uzavrú (`max_gap`),
- potom sa aplikuje morfologická operácia OPEN s horizontálnym kernelom,
- výsledkom sú len tie úseky, ktoré majú dĺžku aspoň `min_run`.

**Význam:**
- stabilizuje masku,
- odstraňuje krátke artefakty,
- zvýrazňuje hlavné frekvenčné línie.


In [11]:
def keep_long_horizontal_runs(mask, min_run=16, max_gap=2):
    m = (mask > 0).astype(np.uint8) * 255
    if max_gap > 0:
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE,
                             cv2.getStructuringElement(cv2.MORPH_RECT, (max_gap+1, 1)), 1)
    k_long = cv2.getStructuringElement(cv2.MORPH_RECT, (int(min_run), 1))
    return cv2.morphologyEx(m, cv2.MORPH_OPEN, k_long, 1)

### Funkcia `robust_pipeline()`

Toto je hlavná čistiaca pipeline, ktorá kombinuje:
- odstránenie šumu (`remove_linear_outliers`),
- filtráciu podľa tvaru (`filter_components_by_shape`),
- zachovanie dlhých horizontálnych úsekov (`keep_long_horizontal_runs`),
- morfologické čistenie a dilatáciu.

**Cieľ:**
Vytvoriť stabilnú, čistú a konzistentnú masku, ktorá obsahuje len relevantné Spread‑F línie.

**Výhody:**
- odolnosť voči šumu,
- zachovanie tenkých línií,
- odstránenie vertikálnych artefaktov,
- príprava masky na následné dopĺňanie medzier.


In [12]:
def robust_pipeline(bin_mask, cfg, img_width):
    m_lin = remove_linear_outliers(
        bin_mask,
        line_len=cfg['line_len'],
        density_thr=cfg['density_thr'],
        v_radius=cfg['v_radius'],
        h_spread=cfg['h_spread'],
        v_thick=cfg['v_thick']
    )
    m_shape = filter_components_by_shape(
        m_lin,
        min_area=cfg['min_area_shape'],
        large_keep=cfg['large_keep'],
        min_elong=cfg['min_elong'],
        max_angle_deg=cfg['max_angle_deg']
    )
    m_runs = keep_long_horizontal_runs(
        m_lin,
        min_run=max(cfg['min_run_px'], img_width // cfg['min_run_div']),
        max_gap=cfg['max_gap']
    )
    banded = cv2.bitwise_or(m_runs, m_shape)
    if cfg.get('rescue_lin', False):
        banded = cv2.bitwise_or(banded, m_lin)
    banded = cv2.morphologyEx(
        banded, cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (cfg['close_w'], cfg['close_h'])),
        1
    )
    banded = cv2.dilate(
        banded,
        cv2.getStructuringElement(cv2.MORPH_RECT, (cfg['dil_w'], cfg['dil_h'])),
        1
    )
    return banded

### Funkcia `find_row_ranges()`

Táto funkcia analyzuje vertikálny profil masky a hľadá **frekvenčné pásma**, v ktorých sa nachádzajú Spread‑F línie.

**Hlavné kroky:**
- vytvorenie vertikálneho histogramu (súčet pixlov v každom riadku),
- vyhladenie histogramu,
- detekcia vrcholov (peakov),
- segmentácia na 1–3 horizontálne pásma,
- rozšírenie pásiem podľa šírky a tvaru,
- korekcia podľa „údolí“ medzi peakmi.

**Výsledok:**
- zoznam dvojíc `(ymin, ymax)`, ktoré definujú jednotlivé frekvenčné pásma.

In [13]:
def find_row_ranges(mask,max_lines=3,smooth_w=None,min_peak_prom=None,min_distance=None,band_drop=None,min_band=None,h_connect=9, v_pad=2,force_three=True, center_window_ratio=0.18, 
                    relax_factor=0.65, valley_split=True, touch_tolerance=0, windowed_fallback=True, q_split=(0.30, 0.70), win_guard=8,widen_pct=0.05):
    import cv2 as _cv2 

    m = (mask > 0).astype(np.uint8)
    H, W = m.shape
    if not m.any():
        return [(0, H-1)]

    mb = m.copy()
    if v_pad > 0:
        mb = _cv2.dilate(mb, _cv2.getStructuringElement(_cv2.MORPH_RECT, (1, 2*v_pad+1)), 1)
    if h_connect and h_connect > 1:
        mb = _cv2.morphologyEx(mb, _cv2.MORPH_CLOSE,
                               _cv2.getStructuringElement(_cv2.MORPH_RECT, (int(h_connect), 1)), 1)

    prof = mb.sum(axis=1).astype(np.float32)
    if prof.max() <= 0:
        return [(0, H-1)]

    if smooth_w is None:
        sw = max(15, int(round(H * 0.02)))
        smooth_w = sw if (sw % 2 == 1) else sw + 1
        smooth_w = min(smooth_w, 61)
    k = np.ones(smooth_w, np.float32) / float(smooth_w)
    prof_s = np.convolve(prof, k, mode='same')

    pmax = float(prof_s.max())
    p = prof_s / (pmax + 1e-6)

    med = float(np.median(p))
    mad = float(np.median(np.abs(p - med)) + 1e-6)

    if min_distance is None:
        min_distance = max(25, int(H * 0.17))
    if min_band is None:
        min_band = max(10, int(H * 0.015))
    if min_peak_prom is None:
        min_peak_prom = float(np.clip(med + 2.0 * mad, 0.05, 0.18))
    if band_drop is None:
        band_drop = float(np.clip(0.36 + 2.0 * mad, 0.32, 0.42))

    fill_ratio = prof.sum() / (H * W + 1e-6)
    if fill_ratio < 0.01:
        min_peak_prom = max(0.035, min_peak_prom * 0.75)
        band_drop = max(0.30, band_drop - 0.03)

    def pick_peaks(p_, mpp, mind, limit=max_lines):
        peaks = []
        for y in range(1, H-1):
            if p_[y] >= p_[y-1] and p_[y] >= p_[y+1] and p_[y] >= mpp:
                peaks.append((p_[y], y))
        peaks.sort(reverse=True)
        sel = []
        for val, y in peaks:
            if all(abs(y - yy) >= mind for _, yy in sel):
                sel.append((val, y))
            if len(sel) >= limit:
                break
        return sel

    selected = pick_peaks(p, min_peak_prom, min_distance)
    if force_three and len(selected) < max_lines:
        selected = pick_peaks(p, max(0.03, min_peak_prom * relax_factor),
                              max(15, int(min_distance * 0.8)))

    if windowed_fallback and len(selected) < max_lines:
        c = np.cumsum(p)
        c /= (c[-1] + 1e-9)
        y1 = int(np.searchsorted(c, q_split[0]))
        y2 = int(np.searchsorted(c, q_split[1]))
        y1 = int(np.clip(y1, win_guard, H - win_guard - 1))
        y2 = int(np.clip(y2, y1 + win_guard, H - win_guard))

        windows = [(0, y1), (y1, y2), (y2, H-1)]
        seeds = []
        for a, b in windows:
            if b - a < 3:
                continue
            yy = a + int(np.argmax(p[a:b+1]))
            val = p[yy]
            if val >= max(0.02, 0.5 * min_peak_prom):
                seeds.append((val, yy))

        cand = sorted(selected + seeds, key=lambda t: -t[0])
        merged = []
        mind = max(12, int(min_distance * 0.6))
        for val, y in cand:
            if all(abs(y - yy) >= mind for _, yy in merged):
                merged.append((val, y))
            if len(merged) >= max_lines:
                break
        if merged:
            selected = merged

    if not selected:
        selected = [(p.max(), int(np.argmax(p)))]

    selected = selected[:max_lines]
    selected.sort(key=lambda t: t[1])

    bands = []
    for val, y0 in selected:
        thr = band_drop * float(val)
        a = y0
        while a > 0 and p[a] >= thr:
            a -= 1
        b = y0
        while b < H-1 and p[b] >= thr:
            b += 1
        bands.append([max(0, a), min(H-1, b), y0])

    if valley_split and len(bands) >= 2:
        bands.sort(key=lambda x: x[2])
        for i in range(len(bands)-1):
            a1, b1, y1 = bands[i]
            a2, b2, y2 = bands[i+1]
            if b1 >= a2 - int(touch_tolerance):
                lo, hi = max(min(y1, y2), a2), min(max(y1, y2), b1)
                if hi <= lo:
                    lo, hi = min(y1, y2), max(y1, y2)
                valley = int(np.argmin(p[lo:hi+1]) + lo)
                bands[i][1]   = max(bands[i][0], valley)
                bands[i+1][0] = min(bands[i+1][1], valley + 1)

    out = []
    for a, b, y0 in bands:
        if (b - a + 1) < min_band:
            half = max(min_band // 2, 1)
            a = max(0, y0 - half)
            b = min(H-1, y0 + half)
        if widen_pct and widen_pct > 0:
            pad = int(round((b - a + 1) * float(widen_pct)))
            a = max(0, a - pad)
            b = min(H-1, b + pad)
        out.append((int(a), int(b)))

    return out[:max_lines] if out else [(0, H-1)]


### Funkcia `repair_top_gaps()`

Táto funkcia opravuje **najväčšie horizontálne medzery** v maske.  
Ide o najdôležitejší krok, ktorý zabezpečuje, že výsledná maska bude mať **spojité frekvenčné línie**.

**Čo robí:**
- rozdelí masku podľa detegovaných pásiem,
- v každom pásme nájde najväčšie medzery,
- pre každú medzeru odhadne:
  - pozíciu ľavého a pravého bodu,
  - výšku línie,
- rozhodne, či použiť:
  - **priamku** (malé medzery),
  - **Z‑tvar** (veľké medzery),
- opraví medzeru nakreslením spojovacej línie.

In [14]:
def repair_top_gaps(mask, row_ranges=None, num_gaps=5, min_gap_for_connection=50, z_threshold=150):
    H, W = mask.shape[:2]
    if row_ranges is None:
        row_ranges = find_row_ranges(mask, max_lines=3)

    bands = [(max(0, int(a)), min(H - 1, int(b))) for (a, b) in row_ranges if b >= a]
    bands.sort(key=lambda ab: (ab[0] + ab[1]) / 2.0)

    cuts = []
    for (a1, b1), (a2, b2) in zip(bands, bands[1:]):
        y = max(0, min(H - 2, (b1 + a2) // 2))
        if cuts and y <= cuts[-1]:
            y = min(H - 2, cuts[-1] + 1)
        cuts.append(y)

    segments = []
    y0 = 0
    for c in cuts:
        segments.append((y0, c))
        y0 = c + 1
    segments.append((y0, H - 1))

    split_rows = [mask[a:b + 1, :] for (a, b) in segments]
    y_offsets = [a for (a, _) in segments]
    if not split_rows:
        return mask.copy()

    gap_lines_by_row = []
    for idx, (row, y_offset) in enumerate(zip(split_rows, y_offsets)):
        row_proj = np.sum(row, axis=0)
        binary_proj = (row_proj > 0).astype(np.int32)
        inverse_proj = 1 - binary_proj
        edges = np.where(np.diff(np.concatenate(([0], inverse_proj, [0]))) != 0)[0]
        gaps = [(edges[i], edges[i+1]) for i in range(0, len(edges)-1, 2)]

        row_gap_data = []
        if gaps:
            lengths = [end - start for start, end in gaps]
            top_idx = np.argsort(lengths)[-num_gaps:][::-1]
            for ti in top_idx:
                start, end = gaps[ti]
                gap_length = end - start
                if gap_length < min_gap_for_connection:
                    continue

                left_start, left_end = max(0, start - 5), max(0, start)
                right_start, right_end = min(row.shape[1], end), min(row.shape[1], end + 5)
                left_slice = row[:, left_start:left_end] if left_end > left_start else row[:, 0:0]
                right_slice = row[:, right_start:right_end] if right_end > right_start else row[:, 0:0]
                left_y, right_y = np.where(left_slice == 255)[0], np.where(right_slice == 255)[0]

                if left_y.size > 0:
                    y1 = int(np.median(left_y))
                else:
                    band_idx = min(idx, len(row_ranges)-1)
                    band_mid = (row_ranges[band_idx][0] + row_ranges[band_idx][1]) // 2
                    y1 = int(band_mid - y_offset)

                if right_y.size > 0:
                    y2 = int(np.median(right_y))
                else:
                    band_idx = min(idx, len(row_ranges)-1)
                    band_mid = (row_ranges[band_idx][0] + row_ranges[band_idx][1]) // 2
                    y2 = int(band_mid - y_offset)

                seg_top, seg_bot = int(y_offset), int(y_offset + row.shape[0] - 1)
                yy1, yy2 = np.clip(int(y1 + y_offset), seg_top, seg_bot), np.clip(int(y2 + y_offset), seg_top, seg_bot)
                p1, p2 = (int(start), yy1), (int(end), yy2)

                row_gap_data.append({
                    'start': int(start), 'end': int(end), 'length': int(gap_length),
                    'p1': p1, 'p2': p2, 'row_index': idx, 'z_candidate': gap_length >= z_threshold
                })
        gap_lines_by_row.append(row_gap_data)

    full_mask = mask.copy()
    grouped_gaps = []
    for row_gaps in gap_lines_by_row:
        for gap in row_gaps:
            matched = False
            for group in grouped_gaps:
                if gaps_overlap(gap, group[0]):
                    group.append(gap)
                    matched = True
                    break
            if not matched:
                grouped_gaps.append([gap])

    repaired_gaps = set()
    for group in grouped_gaps:
        by_row = [None] * len(split_rows)
        for g in group:
            by_row[g['row_index']] = g
        is_all_z = all(g and g['z_candidate'] for g in by_row)
        for g in [x for x in by_row if x]:
            seg_top = y_offsets[g['row_index']]
            seg_bot = seg_top + split_rows[g['row_index']].shape[0] - 1
            if is_all_z:
                full_mask = connect_z_shape_dynamic(full_mask, g['p1'], g['p2'], gap_length=g['length'])
            else:
                full_mask = connect_line_segment(full_mask, g['p1'], g['p2'], seg_top, seg_bot, thickness=2)
            repaired_gaps.add((g['row_index'], g['start'], g['end']))

    for row_idx, row_gaps in enumerate(gap_lines_by_row):
        for g in row_gaps:
            if (g['row_index'], g['start'], g['end']) not in repaired_gaps:
                seg_top = y_offsets[g['row_index']]
                seg_bot = seg_top + split_rows[g['row_index']].shape[0] - 1
                full_mask = connect_line_segment(full_mask, g['p1'], g['p2'], seg_top, seg_bot, thickness=2)
    return full_mask

### Hlavná slučka — spracovanie všetkých obrázkov

Táto bunka vykonáva kompletnú pipeline pre každý vstupný obrázok:

- **Načítanie obrázka**
- **Farebná extrakcia (HSV + LAB)**
   - detekcia červených, oranžových a hnedých odtieňov,
   - odstránenie príliš svetlých a sivých oblastí.
- **Počiatočná maska**
- **Robustná filtrácia masky**
- **Detekcia frekvenčných pásiem**
- **Predĺženie línií pri okrajoch**
- **Oprava veľkých medzier**
- **Uloženie výslednej masky**
- **Vizualizácia (originál, extrakcia, filtrácia, maska, overlay)**

Táto časť notebooku predstavuje kompletný proces generovania algoritmických masiek.


In [ ]:
fig, axs = plt.subplots(len(filenames), 5, figsize=(30, 6 * max(1, len(filenames))))
if len(filenames) == 1:
    axs = np.expand_dims(axs, 0)

for i, fname in enumerate(filenames):
    image = cv2.imread(os.path.join(input_folder, fname))
    if image is None:
        print(f"Obrázok {fname} sa nepodarilo načítať.")
        continue

    is_hard = ('!' in fname)
    H, W = image.shape[:2]

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    Hh, Ss, Vv = cv2.split(hsv)
    
    lower_red1 = np.array([0,   40,  22], np.uint8)
    upper_red1 = np.array([16, 255, 255], np.uint8)
    lower_red2 = np.array([164, 40,  22], np.uint8)
    upper_red2 = np.array([180,255, 255], np.uint8)
    
    lower_orange = np.array([5,  35,  22], np.uint8)
    upper_orange = np.array([50, 255, 255], np.uint8)
    
    lower_brown  = np.array([5,  22,  15], np.uint8)
    upper_brown  = np.array([30, 255, 220], np.uint8)
    
    mask_red    = cv2.inRange(hsv, lower_red1, upper_red1) | cv2.inRange(hsv, lower_red2, upper_red2)
    mask_orange = cv2.inRange(hsv, lower_orange, upper_orange)
    mask_brown  = cv2.inRange(hsv, lower_brown,  upper_brown)
    mask_warm   = mask_red | mask_orange | mask_brown
    
    too_bright = (Vv > 254).astype(np.uint8) * 255
    too_gray   = (Ss <  42).astype(np.uint8) * 255   
    mask_warm  = cv2.bitwise_and(mask_warm, cv2.bitwise_not(too_bright))
    mask_warm  = cv2.bitwise_and(mask_warm, cv2.bitwise_not(too_gray))
    
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    lab_gate = cv2.inRange(
        lab,
        np.array([8, 108,  98], np.uint8),   # L↓, a/b↓
        np.array([255,255,255], np.uint8)
    )
    initial_mask = cv2.bitwise_and(mask_warm, lab_gate)
    
    
    # ľahké „prilepenie“ tenkých ťahov, aby sa nestrácali
    initial_mask = cv2.morphologyEx(
        initial_mask, cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (3,1)), 1
    )


    cfg_soft = dict(
        line_len=79, density_thr=0.26, v_radius=9, h_spread=141, v_thick=1,
        min_area_shape=6, large_keep=600, min_elong=2.4, max_angle_deg=28,
        min_run_px=10, min_run_div=96, max_gap=3,
        close_w=5, close_h=1, dil_w=2, dil_h=1,
        rescue_lin=True
    )
    cfg_hard = dict(
        line_len=99, density_thr=0.34, v_radius=6, h_spread=151, v_thick=1,
        min_area_shape=10, large_keep=600, min_elong=3.2, max_angle_deg=22,
        min_run_px=16, min_run_div=64, max_gap=2,
        close_w=9, close_h=1, dil_w=3, dil_h=2
    )
    cfg = cfg_hard if is_hard else cfg_soft
    banded = robust_pipeline(initial_mask, cfg, img_width=W)

    row_ranges = find_row_ranges(initial_mask, max_lines=3, min_distance=40)


    H, W = image.shape[:2]
    bands = []
    for (ymin, ymax) in row_ranges:
        a = max(0, int(ymin)); b = min(H-1, int(ymax))
        if b >= a:
            bands.append((a, b))
    bands.sort(key=lambda ab: (ab[0] + ab[1]) / 2.0)
    
    cuts = []
    for (a1, b1), (a2, b2) in zip(bands, bands[1:]):
        y = (b1 + a2) // 2
        if cuts and y <= cuts[-1]:
            y = min(H-2, cuts[-1] + 1)
        cuts.append(int(y))

    extended_mask = extend_line_ends_near_edges(
        banded.copy(),
        row_ranges=row_ranges,
        edge_max_gap=50,
        endpoint_band=5,
        thickness=4,
        min_area=30
    )

    min_gap_for_connection = 12 if is_hard else 10
    z_threshold = 90 if is_hard else 60

    
    repaired_mask_with_shape = repair_top_gaps(
        extended_mask.copy(),
        row_ranges=row_ranges,
        num_gaps=100,
        min_gap_for_connection=min_gap_for_connection,
        z_threshold=40
    )

    # Saving the mask
    output_mask_path = os.path.join(output_folder, os.path.splitext(fname)[0] + '_mask.png')
    cv2.imwrite(output_mask_path, repaired_mask_with_shape)

    # Visualization
    axs[i, 0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axs[i, 0].set_title('Original image \n' + fname, fontsize=20); axs[i, 0].axis('off')

    axs[i, 1].imshow(initial_mask, cmap='gray')
    axs[i, 1].set_title('Extraction (red+yellow)', fontsize=20); axs[i, 1].axis('off')

    axs[i, 2].imshow(extended_mask, cmap='gray')
    axs[i, 2].set_title('Final noise removal', fontsize=20); axs[i, 2].axis('off')

    axs[i, 3].imshow(repaired_mask_with_shape, cmap='gray')
    axs[i, 3].set_title('Created mask', fontsize=20); axs[i, 3].axis('off')

    overlay_image = cv2.cvtColor(image.copy(), cv2.COLOR_BGR2RGB)
    mask_colored = np.zeros_like(overlay_image)
    mask_colored[:, :, 0] = repaired_mask_with_shape  
    overlay = cv2.addWeighted(overlay_image, 0.7, mask_colored, 0.6, 0)
    
    axs[i, 4].imshow(overlay)
    axs[i, 4].set_title('The mask in the picture', fontsize=20); axs[i, 4].axis('off')

plt.tight_layout()
plt.show()
